# NOT training with GDmax and Extragradient

In [ ]:
import os, sys
sys.path.append("..")

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import clear_output

import math
device = "cuda" if torch.cuda.is_available() else "cpu"

from src.models import Transport, Critic
from src.utils import cost, show_mapping , grad_norm, cosine_lr, plot_3d_function, ema_np, plot_loss, plot_grad_norm
from src.train import train_gdmax, train_extragradient

EXP_DIR = "../outputs/toy_gdmax"
GDMAX_DIR = os.path.join(EXP_DIR, "gdmax")
EG_DIR = os.path.join(EXP_DIR, "eg")
os.makedirs(GDMAX_DIR, exist_ok=True)
os.makedirs(EG_DIR, exist_ok=True)


In [ ]:
# source distribution μ
def sample_mu(batch_size, device="cpu"):
    x = 2 * torch.rand(batch_size, 1, device=device) - 1  # [-1,1]
    y = torch.zeros(batch_size, 1, device=device)         # y = 0
    return torch.cat([x, y], dim=1)


# target distribution ν
def sample_nu(batch_size, device="cpu"):
    x = torch.zeros(batch_size, 1, device=device)         # y = 0
    y = 2 * torch.rand(batch_size, 1, device=device) - 1  # [-1,1]
    return torch.cat([x, y], dim=1)

## GDmax Training

In [ ]:
T_gdmax = Transport().to(device)
f_gdmax = Critic().to(device)

history_gdmax = train_gdmax(
    T_gdmax, f_gdmax, sample_mu, sample_nu, cost,
    n_steps=20000, lr_T=1e-4, lr_f=1e-4, K=10,
    batch_size_x=1024, batch_size_y=1024,
    lr_schedule=lambda step: (cosine_lr(step, 2000, 1e-4, 0),) * 2,
    log_dir=GDMAX_DIR,
    callback=lambda step, T, f: show_mapping(
        T, sample_mu, sample_nu,
        save_path=os.path.join(GDMAX_DIR, f"step_{step + 1:06d}.png"),
    ),
)

In [ ]:
plot_loss(history_gdmax["loss"], save_path=os.path.join(GDMAX_DIR, "loss.png"))
plot_grad_norm(history_gdmax["grad_T"], history_gdmax["grad_f"], save_path=os.path.join(GDMAX_DIR, "grad_norm.png"))
show_mapping(T_gdmax, sample_mu, sample_nu, option=True, save_path=os.path.join(GDMAX_DIR, "final_mapping.png"))
plot_3d_function(f_gdmax, save_path=os.path.join(GDMAX_DIR, "final_f.png"))

## Extragradient Training

In [ ]:
torch.manual_seed(1000)  # known-stable seed for raw-gradient EG at lr=1e-3
T_eg = Transport().to(device)
f_eg = Critic().to(device)

history_eg = train_extragradient(
    T_eg, f_eg, sample_mu, sample_nu, cost,
    n_steps=100000, lr_T=1e-3, lr_f=1e-3,
    batch_size_x=1024, batch_size_y=1024,
    lr_schedule=lambda step: (cosine_lr(step, 10000, 1e-3, 0),) * 2,
    log_dir=EG_DIR,
    callback=lambda step, T, f: show_mapping(
        T, sample_mu, sample_nu,
        save_path=os.path.join(EG_DIR, f"step_{step + 1:06d}.png"),
    ),
)

In [ ]:
plot_loss(history_eg["loss"], save_path=os.path.join(EG_DIR, "loss.png"))
plot_grad_norm(history_eg["grad_T"], history_eg["grad_f"], save_path=os.path.join(EG_DIR, "grad_norm.png"))
show_mapping(T_eg, sample_mu, sample_nu, option=True, save_path=os.path.join(EG_DIR, "final_mapping.png"))
plot_3d_function(f_eg, save_path=os.path.join(EG_DIR, "final_f.png"))